In [6]:
!pip install torch torchvision torchaudio
!pip install diffusers transformers accelerate peft
!pip install   pillow numpy opencv-python datasets
!pip install git+https://github.com/huggingface/diffusers
!pip install streamlit

# Required libraries for project.

  Cloning https://github.com/huggingface/diffusers to /private/var/folders/90/7bhc4qw17dd0tc8r6137_d040000gn/T/pip-req-build-minz71bb
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers /private/var/folders/90/7bhc4qw17dd0tc8r6137_d040000gn/T/pip-req-build-minz71bb
  Resolved https://github.com/huggingface/diffusers to commit 4a2833c1c226362677e165a363fecb4da331e122
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# !pip install huggingface_hub


from huggingface_hub import snapshot_download
# from hugging face download the dataset :
# dataset is Disney video generation dataset.
# snapshot_download() downloads an entire repository at a given revision. It uses internally hf_hub_download() which means all downloaded files are also cached on your local disk. 

snapshot_download(
    repo_id="Wild-Heart/Disney-VideoGeneration-Dataset",
    repo_type="dataset",
    local_dir="./video_dataset"
)

ModuleNotFoundError: No module named 'huggingface_hub'

In [ ]:
# import the processor class for BLIP

from transformers import BlipProcessor, BlipForConditionalGeneration
#Pillow(PIL) :  Python library for image processing, allowing users to open, manipulate, and save various file formats 
from PIL import Image
# BLIP model runs using torch tensors.
import torch

# BLIP (Bootstrapped Language-Image Pretraining) is a vision-language pretraining (VLP) framework designed 
# for both understanding and generation tasks. Most existing pretrained models are only good at one or the other. 
# It uses a captioner to generate captions and a filter to remove the noisy captions. This increases training data quality and 
# more effectively uses the messy web data.


# import the processor class for BLIP
# this handle image preprocessing + test tokenization automatically.
processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
# "Salesforce/blip-image-captioning-base" is model id. 
# BLIP model runs using torch tensors.


# BlipForConditionalGeneration. is used because image captioning is a sequence generation task.
# it generate text (caption) conditioned on an image.

model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to("mps")


def generate_caption(image):
    # returns pytorch tensors. for image.
    inputs = processor(image, return_tensors="pt").to("mps")


    # autoregressive text generation.
    # inputs is the input text arguments.
    #model predict the next token.
    # until it generates a complete caption.
    out = model.generate(**inputs)


    # convert token IDs back to readable text.
    # skip_special tokens.
    return processor.decode(out[0], skip_special_tokens=True)

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 2285.37it/s, Materializing param=vision_model.post_layernorm.weight]                                       
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT 

In [ ]:
# opencv is used to read video files frame by frame.
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset



class VideoDataset(Dataset):
    def __init__(self, video_paths, captions, size=256, frames=16):
        self.video_paths = video_paths
        self.captions = captions
        self.size = size
        self.frames = frames

    def __len__(self):
        return len(self.video_paths)

    def load_video(self, path):
        # create a video capture object that allows frame-by-frame reading.
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # sample frame indices uniformly
        indices = np.linspace(0, total - 1, self.frames).astype(int)

        frames = []
        # current_frame track position in video
        current_frame = 0
        # target_id tracks which sampled index we want next.
        target_id = 0

        while cap.isOpened():
            ret, frame = cap.read()
            # ret is for whether reading is succeeded or not
            # frame is an image array.
            if not ret:
                break
            # if reading is not succeeded then just return
            #otherwise read the frame and append this frame to the frames array.

            if target_id < len(indices) and current_frame == indices[target_id]:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
                target_id += 1

            current_frame += 1

        cap.release() 
        # free the memory for next frame.

        

        
        if len(frames) == 0:
            raise ValueError(f"Failed to load any frames from video: {path}. check the path please.")

        

        # convert list of frames to numpy array. 
        # normalized the values and scaled then between 0 to 1.
        frames = np.array(frames).astype(np.float32) / 255.0

        # original shape we get is (frames,height,width,channels)
        # but  deep learning expects (frames,chanels,height,width)
        # so we reorder the each frame in frames list.
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2)



        # interpolation used for resizing input tensors.
        # what is the input size, resize the input into required output size and bilinear mode used for smooth resizing.

        frames = F.interpolate(
            frames,
            size=(self.size, self.size),
            mode="bilinear",
            align_corners=False
        )

        return frames

    def __getitem__(self, idx):

        frames = self.load_video(self.video_paths[idx])
    # hugging face training format.
        return {
            "pixel_values": frames,
            "caption": self.captions[idx]
        }

In [ ]:
# import cv2
# import torch
# import numpy as np
# import torch.nn.functional as F
# from torch.utils.data import Dataset


# class VideoDataset(Dataset):
#     def __init__(self, video_paths, captions, size=256, frames=16):
#         self.video_paths = video_paths
#         self.captions = captions
#         self.size = size
#         self.frames = frames

#     def __len__(self):
#         return len(self.video_paths)

#     def __getitem__(self, idx):

#         path = self.video_paths[idx]
#         cap = cv2.VideoCapture(path)

#         total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

#         # same as np.linspace in your decord version
#         indices = np.linspace(0, total - 1, self.frames).astype(int)

#         frames = []

#         for frame_id in indices:
#             cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
#             ret, frame = cap.read()

#             if not ret:
#                 continue

#             frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             frames.append(frame)

#         cap.release()

#         # convert to tensor like decord output
#         frames = np.array(frames).astype(np.float32) / 255.0
#         frames = torch.from_numpy(frames).permute(0, 3, 1, 2)

#         # SAME interpolate step as your code
#         frames = F.interpolate(
#             frames,
#             size=(self.size, self.size),
#             mode="bilinear",
#             align_corners=False
#         )

#         return {
#             "pixel_values": frames,
#             "caption": self.captions[idx]
#         }

In [ ]:
# with open("./video_dataset/videos.txt") as f:
#     videos = [f"./video_dataset/{x}" for x in f]

# with open("./video_dataset/prompt.txt") as f:
#     captions = [x.strip() for x in f]

# # print(captions)
with open("./video_dataset/videos.txt",'r') as f:
    # Use .strip() to remove the \n, and remove the extra 'videos/' 
    # since it's already in the text file
    videos = [f"./video_dataset/{x.strip()}" for x in f]

with open("./video_dataset/prompt.txt") as f:
    captions = [x.strip() for x in f]

# print(videos)
# with open("video.txt", "r") as f:
#     video_list = [line.strip() for line in f]

# print(video_list)
for caption in captions:
    print(caption,end='\n')

A black and white animated scene unfolds with an anthropomorphic goat surrounded by musical notes and symbols, suggesting a playful environment. Mickey Mouse appears, leaning forward in curiosity as the goat remains still. The goat then engages with Mickey, who bends down to converse or react. The dynamics shift as Mickey grabs the goat, potentially in surprise or playfulness, amidst a minimalistic background. The scene captures the evolving relationship between the two characters in a whimsical, animated setting, emphasizing their interactions and emotions.
A black and white animated sequence on a ship's deck features an anthropomorphic bulldog character, showcasing exaggerated facial expressions and body language. The character progresses from confident to focused, then to strained and distressed, displaying a range of emotions as it navigates challenges. The ship's interior remains static in the background, with minimalistic details such as a bell and open door. The character's dyna

In [ ]:
# del pipe
# import gc
# gc.collect()

# import torch
# torch.mps.empty_cache()

from diffusers import StableVideoDiffusionPipeline

import torch
# from_pretrained  downloads pretrained stable video diffusion weights from huggingFace.
# dtype=fp16 reduces memory usage.
# 
pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16
)
# move entire model to Apple GPU
pipe.to("mps")

#DDPM(Denoising Diffusion Probabilistic Models)  refers to the discrete denoising scheduler from the paper as well as pipeline.
from diffusers import DDPMScheduler

# Swap the inference scheduler for a training-friendly DDPM scheduler
pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# 1. Setup - Accelerator handle the precision!
device = "mps"




# import torch, gc
# from diffusers import StableVideoDiffusionPipeline, DDPMScheduler

# # Clean memory
# try:
#     del pipe
# except:
#     pass

# gc.collect()
# torch.mps.empty_cache()

# # Load pipeline
# pipe = StableVideoDiffusionPipeline.from_pretrained(
#     "stabilityai/stable-video-diffusion-img2vid-xt",
#     torch_dtype=torch.float16,
#     variant="fp16"
# )

# # ✅ DO NOT use pipe.to("mps")
# pipe.enable_model_cpu_offload()
# pipe.enable_attention_slicing()
# pipe.enable_vae_slicing()

# # Scheduler swap (this part is correct)
# pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# device = "mps"


# del pipe
# import gc, torch
# gc.collect()
# torch.mps.empty_cache()

# from diffusers import StableVideoDiffusionPipeline

# pipe = StableVideoDiffusionPipeline.from_pretrained(
#     "stabilityai/stable-video-diffusion-img2vid-xt",
#     torch_dtype=torch.float16,
#     variant="fp16"
# )

# Use this instead of pipe.to("mps")
# pipe.enable_model_cpu_offload()
# pipe.enable_attention_slicing()
# pipe.enable_vae_slicing()

Loading pipeline components...: 100%|██████████| 5/5 [00:11<00:00,  2.33s/it]


In [ ]:
# Peft _> parameter efficient fine tunning.   instead of training the entire huge model we traines only a small low rank adapters.
# this saves memory.
# this speeds up the training process
# prevents overfitting
# works great on small Gpu

# Lora: low rank adapter

from peft import LoraConfig, get_peft_model

# we are using here lora with rank = 8 
# lora replace weight update -> w=w+BA
# where A is rxd
# B is dxr

# lora alpha 
# final lora update:
# w=w+alpha*(BA)/r

# to_q=Query projection
# to_key=key prrojection
# value projection=>to_v
# output projection=>to_out

# initailize the lora weights from guassian distribution.

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["to_q","to_k","to_v","to_out.0"],
    init_lora_weights="gaussian"
)

# apply lora to unet.
# unet is a convolutional neural netowrk(CNN) architecture designed for fast,precise image segmentation.
# it utilizes U-shaped structure consisting of a contracting path to capture context and a symmetric exapnding path for  precise localization connect by skip connections that transfer high-resolution features.

# takes existing UNet.
# injects Lora layers into specified modules.
# freezes original weights
# makes only lora weights trainable.


pipe.unet = get_peft_model(pipe.unet, lora_config)

In [ ]:
pipe.unet.enable_gradient_checkpointing()
# forward pass compute activations
# activations are stored in memory
# backward pass uses stored activations to compute gradients.

pipe.enable_model_cpu_offload()
# loading entire model on GPU


In [3]:
# data loader is used to batch and shuffle dataset samples.
# Auto tokenizer converts text captions into token IDS
# accelerate simplifies mixed precision and device handling.

from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from accelerate import Accelerator

dataset = VideoDataset(videos, captions, size=256, frames=16)
# create a dataset using vides and captions.


loader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True
)

accelerator = Accelerator(
    mixed_precision="fp16"
)
# AdamW optimizer is used for transformer based models.
# lr = learning rate.
# use unet parameters.
optimizer = torch.optim.AdamW(
    pipe.unet.parameters(),
    lr=1e-4
)

pipe.unet, optimizer, loader = accelerator.prepare(
    pipe.unet,
    optimizer,
    loader
)

ModuleNotFoundError: No module named 'torch'

In [4]:

import torch
import torch.nn.functional as F
#maximum training step.
# global counter step to stop training 
num_steps = 300
global_step = 0

device = "mps"

# vae -> variational encoder are probabilistic generative deep learning models that encode input data into a continous , structured latent space to reconstruct , denoise, or generate new similiar data.

pipe.vae.to(device)
pipe.image_encoder.to(device)

# freeze the VAE weights.
# freeze image encoder weights.

pipe.vae.requires_grad_(False) 
pipe.image_encoder.requires_grad_(False)

for epoch in range(3):
    for batch in loader:
        with accelerator.accumulate(pipe.unet):
            
            # pixel_values shape: [bs, frames, channels, height, width]
            # Assuming values are [0, 1]. SVD VAE expects [-1, 1].so we normalized the data.

            pixel_values = batch["pixel_values"].to(device=device, dtype=torch.float16)
            pixel_values_scaled = pixel_values * 2.0 - 1.0 
            
            bs, num_frames, c, h, w = pixel_values.shape

            # 1.Extract First Frame for Conditioning
            first_frames = pixel_values_scaled[:, 0, :, :, :] # [bs, c, h, w]

            # 2.Get Image Embeddings (SVD requires this instead of text)
            # Resize for CLIP Vision model (224x224)
            image_for_clip = F.interpolate(first_frames, size=(224, 224), mode='bilinear')
            with torch.no_grad():
                image_embeddings = pipe.image_encoder(image_for_clip).image_embeds
                # SVD expects sequence length of 1 for the embedding
                image_embeddings = image_embeddings.unsqueeze(1) # [bs, 1, 1024]

            # Encode Video with VAE 
            pixel_values_flat = pixel_values_scaled.view(bs * num_frames, c, h, w)
            with torch.no_grad():
                latents = pipe.vae.encode(pixel_values_flat).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
            latents = latents.view(bs, num_frames, 4, h // 8, w // 8)

            # Encode First Frame Latents (Image Conditioning) 
            with torch.no_grad():
                cond_latents = pipe.vae.encode(first_frames).latent_dist.sample()
            cond_latents = cond_latents * pipe.vae.config.scaling_factor
            # Repeat the single condition frame to match the number of video frames
            cond_latents = cond_latents.unsqueeze(1).repeat(1, num_frames, 1, 1, 1)

            #  Sample Noise 
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, pipe.scheduler.config.num_train_timesteps, (bs,), device=device
            ).long()

            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)

            # SVD UNet takes 8 channels: 4 (noisy video latents) + 4 (clean first-frame latents)
            model_input = torch.cat([noisy_latents, cond_latents], dim=2) 

            # Prepare Time IDs 
            # SVD specific time ids: [fps, motion_bucket_id, noise_aug_level]
            added_time_ids = torch.tensor(
                [[7.0, 127.0, 0.02]], 
                device=device
            ).repeat(bs, 1)

            # Forward pass 
            model_pred = pipe.unet(
                model_input,
                timesteps,
                encoder_hidden_states=image_embeddings,
                added_time_ids=added_time_ids
            ).sample

            # Loss 
            loss = torch.nn.functional.mse_loss(model_pred.float(), noise.float(), reduction="mean")

            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

            global_step += 1

            if global_step % 10 == 0:      
                print(f"Step: {global_step} | Loss: {loss.item():.4f}")

            if global_step >= 300:
                break
    if global_step >= 300:
        break

ModuleNotFoundError: No module named 'torch'

In [5]:
pipe.unet.save_pretrained("./svd_video_lora")

NameError: name 'pipe' is not defined

In [15]:
from diffusers import StableVideoDiffusionPipeline
from peft import PeftModel

pipe = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=torch.float16
).to("mps")

pipe.unet = PeftModel.from_pretrained(
    pipe.unet,
    "./svd_video_lora"
)

Loading pipeline components...: 100%|██████████| 5/5 [00:12<00:00,  2.48s/it]


In [16]:

from PIL import Image

# 1. Provide a starting image
# Option A: Load an image from your dataset
# init_image = Image.open("path_to_your_disney_character_image.jpg").resize((256, 256))

# Option B: Create a dummy solid-color image just to test the pipeline runs
init_image = Image.new("RGB", (256, 256), color=(50, 50, 150))

# 2. Pass the 'image' argument instead of 'prompt'
frames = pipe(
    image=init_image,
    num_frames=16,
    height=256,
    width=256,
    decode_chunk_size=8, # Pro-tip: Add this to save VRAM during decoding!
).frames[0]


100%|██████████| 25/25 [04:30<00:00, 10.83s/it]


In [17]:
# # # !pip install gradio
# # import gradio as gr

# # def generate(prompt):
# #     vid = pipe(prompt=prompt, num_frames=16).frames[0]
# #     return vid

# # gr.Interface(
# #     fn=generate,
# #     inputs="text",
# #     outputs="video"
# # ).launch()

# # 1. Install required packages (Gradio and ffmpeg for saving mp4 files)
# !pip install gradio imageio[ffmpeg]

# import gradio as gr
# import torch
# import gc
# from diffusers import StableDiffusionPipeline
# from diffusers.utils import export_to_video

# # 2. Load the Text-to-Image pipeline globally (so it doesn't reload on every click)
# t2i_pipe = StableDiffusionPipeline.from_pretrained(
#     "runwayml/stable-diffusion-v1-5", 
#     torch_dtype=torch.float16
# )

# def generate_video(prompt):
#     print("🎨 Generating initial image...")
#     # Move T2I model to GPU
#     t2i_pipe.to("mps")
    
#     # Generate and resize image to fit SVD expectations
#     init_image = t2i_pipe(prompt).images[0]
#     init_image = init_image.resize((256, 256))
    
#     # Offload T2I model and clear VRAM immediately
#     t2i_pipe.to("cpu")
#     gc.collect()
#     torch.mps.empty_cache()
    
#     print("🎬 Generating video frames...")
#     # Move SVD model to GPU
#     pipe.to("mps")
    
#     # Generate video frames from the initial image
#     video_frames = pipe(
#         image=init_image,
#         num_frames=16,
#         height=256,
#         width=256,
#         decode_chunk_size=4 # Crucial for saving VRAM on Mac!
#     ).frames[0]
    
#     # Offload SVD model and clear VRAM
#     pipe.to("cpu")
#     gc.collect()
#     torch.mps.empty_cache()
    
#     print("💾 Saving video...")
#     # Export frames to a file and return the file path for Gradio
#     output_path = "gradio_output.mp4"
#     export_to_video(video_frames, output_path, fps=7)
    
#     return output_path

# # 3. Build and launch the interface
# demo = gr.Interface(
#     fn=generate_video,
#     inputs=gr.Textbox(label="Enter your prompt", placeholder="cinematic disney style character walking in rain..."),
#     outputs=gr.Video(label="Generated Video"),
#     title="Disney-Style Text-to-Video",
#     description="Generates a 16-frame video using SD 1.5 and your fine-tuned Stable Video Diffusion LoRA."
# )

# demo.launch()
# 1. Install required packages (Notice the quotes around "imageio[ffmpeg]"!)
# !pip install  "imageio[ffmpeg]"
# from IPython.display import Video, display
# import gradio as gr
# import torch
# import gc
# from diffusers import StableDiffusionPipeline
# from diffusers.utils import export_to_video

# # 2. Load the Text-to-Image pipeline globally (so it doesn't reload on every click)
# t2i_pipe = StableDiffusionPipeline.from_pretrained(
#     "runwayml/stable-diffusion-v1-5", 
#     torch_dtype=torch.float16
# )

# def generate_video(prompt):
#     print("🎨 Generating initial image...")
#     # Move T2I model to GPU
#     t2i_pipe.to("mps")
    
#     # Generate and resize image to fit SVD expectations
#     init_image = t2i_pipe(prompt).images[0]
#     init_image = init_image.resize((256, 256))
    
#     # Offload T2I model and clear VRAM immediately
#     t2i_pipe.to("cpu")
#     gc.collect()
#     torch.mps.empty_cache()
    
#     print("🎬 Generating video frames...")
#     # Move SVD model to GPU
#     pipe.to("mps")
    
#     # Generate video frames from the initial image
#     video_frames = pipe(
#         image=init_image,
#         num_frames=16,
#         height=256,
#         width=256,
#         decode_chunk_size=4 # Crucial for saving VRAM on Mac!
#     ).frames[0]
    
#     # Offload SVD model and clear VRAM
#     pipe.to("cpu")
#     gc.collect()
#     torch.mps.empty_cache()
    
#     print("💾 Saving video...")
#     # Export frames to a file and return the file path for Gradio
#     output_path = "gradio_output.mp4"
#     export_to_video(video_frames, output_path, fps=7)
    
#     return output_path

# # # 3. Build and launch the interface
# # demo = gr.Interface(
# #     fn=generate_video,
# #     inputs=gr.Textbox(label="Enter your prompt", placeholder="cinematic disney style character walking in rain..."),
# #     outputs=gr.Video(label="Generated Video"),
# #     title="Disney-Style Text-to-Video",
# #     description="Generates a 16-frame video using SD 1.5 and your fine-tuned Stable Video Diffusion LoRA."
# # )

# # demo.launch()
# display(Video(output_mp4_path, embed=True, width=512))

from IPython.display import Video, display
import torch
import gc
from diffusers import StableDiffusionPipeline
from diffusers.utils import export_to_video


# Load Text-to-Image model

t2i_pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)



def generate_video(prompt):

    print("Generating initial image")

    
    t2i_pipe.to("mps")

    init_image = t2i_pipe(prompt).images[0]
    init_image = init_image.resize((256, 256))

    # Free GPU memory
    t2i_pipe.to("cpu")
    gc.collect()
    torch.mps.empty_cache()

    print("Generating video frames")

    
    pipe.to("mps")

    video_frames = pipe(
        image=init_image,
        num_frames=64,
        height=256,
        width=256,
        decode_chunk_size=4   # VERY important on Mac
    ).frames[0]

#     User Prompt
#       ↓
#   Stable Diffusion (t2i_pipe)
#       ↓
#   Initial Image
#       ↓
#    Stable Video Diffusion (pipe)
#       ↓
#   Generated Video

    # Free GPU again
    pipe.to("cpu")
    gc.collect()
    torch.mps.empty_cache()

    print("Saving video")

    import imageio
    import numpy as np

    output_path = "gradio_output.mp4"

    frames_np = [np.array(frame).astype(np.uint8) for frame in video_frames]

    with imageio.get_writer(output_path, fps=7) as writer:
        for frame in frames_np:
            writer.append_data(frame)

    return output_path


# output_mp4_path = generate_video(
#     "cinematic disney style character walking in forest"
# )

# display(Video(output_mp4_path, embed=True, width=512))
import streamlit as st

st.title("🎬 Text to Video Generator")

# Text input for prompt
prompt = st.text_input("Enter your prompt:", "cinematic disney style character walking in forest")

# Generate button
if st.button("Generate Video"):
    with st.spinner("Generating video... ⏳"):
        output_mp4_path = generate_video(prompt)

    st.success("Video generated!")

    # Display video
    st.video(output_mp4_path)

Loading weights: 100%|██████████| 396/396 [00:00<00:00, 396.72it/s, Materializing param=visual_projection.weight]
StableDiffusionSafetyChecker LOAD REPORT from: /Users/pagupta/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 196/196 [00:00<00:00, 345.94it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: /Users/pagupta/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
---------------

SyntaxError: invalid syntax (2260649391.py, line 1)

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu   # adjust for your platform
!pip install diffusers transformers accelerate peft datasets pillow numpy opencv-python decord imageio[ffmpeg] lpips gradio
!pip install git+https://github.com/huggingface/diffusers.git

Looking in indexes: https://download.pytorch.org/whl/cpu
zsh:1: no matches found: imageio[ffmpeg]
  Cloning https://github.com/huggingface/diffusers.git to /private/var/folders/90/7bhc4qw17dd0tc8r6137_d040000gn/T/pip-req-build-9y_2ft69
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/diffusers.git /private/var/folders/90/7bhc4qw17dd0tc8r6137_d040000gn/T/pip-req-build-9y_2ft69
  Resolved https://github.com/huggingface/diffusers.git to commit 1fe688a651bc078326082b8927f8fbdd6cefeef0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
# !pip install lpips


import os
import gc
import cv2
import torch
import lpips
import numpy as np
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import BlipProcessor, BlipForConditionalGeneration, AutoTokenizer
from diffusers import StableVideoDiffusionPipeline, DDPMScheduler
from diffusers.utils import export_to_video
from peft import LoraConfig, get_peft_model, PeftModel
from accelerate import Accelerator
import gradio as gr
import imageio
from IPython.display import Video, display

# -------------------- Configuration --------------------
class Config:
    # Data
    video_dir = "./my_video_clips"          # place your .mp4 clips here
    caption_file = "./captions.txt"          # will be created by BLIP
    size = 256
    num_frames = 16
    fps = 7
    
    # Model
    base_model = "stabilityai/stable-video-diffusion-img2vid-xt"
    lora_rank = 8
    lora_alpha = 8
    target_modules = ["to_q", "to_k", "to_v", "to_out.0"]
    
    # Training
    batch_size = 1
    learning_rate = 1e-4
    max_train_steps = 3000                    # 1k-5k as required
    gradient_accumulation_steps = 1
    mixed_precision = "fp16"
    seed = 42
    
    # Inference
    inference_prompt = "cinematic disney style character walking in forest"

config = Config()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 12.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [lpips]


/Users/pagupta/Desktop/GD_Projects/module_2_all_courses/ML_Specialization_coursera/ml_special_311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'gradio'

In [ ]:
def generate_captions(video_dir, output_file):
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("mps")
    
    video_paths = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.endswith(('.mp4', '.avi', '.mov'))]
    captions = []
    
    for path in video_paths:
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        mid_frame = total_frames // 2
        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            captions.append("")
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        inputs = processor(pil_image, return_tensors="pt").to("mps")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        captions.append(caption)
    
    with open(output_file, 'w') as f:
        for cap in captions:
            f.write(cap + "\n")
    
    return video_paths, captions

# Run this only once to create caption file
# video_paths, captions = generate_captions(config.video_dir, config.caption_file)


In [ ]:
#!/usr/bin/env python
# coding: utf-8

"""
Fine‑tune Stable Video Diffusion on Disney Video Dataset with LoRA.
Run this script cell by cell in a Jupyter notebook or execute as a Python script.
"""

# %% [markdown]
# # 1. Install Required Libraries (run once)

# %%
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu   # adjust for your platform
# !pip install diffusers transformers accelerate peft datasets pillow numpy opencv-python decord imageio[ffmpeg] lpips gradio torchmetrics
# !pip install git+https://github.com/huggingface/diffusers.git

# %% [markdown]
# # 2. Imports and Configuration

# %%
# !pip install gradio torchmetrics

import os
import gc
import cv2
import torch
import lpips
import numpy as np
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import BlipProcessor, BlipForConditionalGeneration, AutoTokenizer
from diffusers import StableVideoDiffusionPipeline, DDPMScheduler, StableDiffusionPipeline
from diffusers.utils import export_to_video
from peft import LoraConfig, get_peft_model, PeftModel
from accelerate import Accelerator
import gradio as gr
import imageio
from huggingface_hub import snapshot_download
from torchmetrics.multimodal.clip_score import CLIPScore
from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
from IPython.display import Video, display

# %%
class Config:
    """All hyperparameters and paths."""
    # Dataset
    dataset_repo = "Wild-Heart/Disney-VideoGeneration-Dataset"
    local_dir = "./video_dataset"
    
    # Data processing
    size = 256
    num_frames = 16
    fps = 7
    
    # Model
    base_model = "stabilityai/stable-video-diffusion-img2vid-xt"
    lora_rank = 8                     # try 4, 8, 16
    lora_alpha = 8
    target_modules = ["to_q", "to_k", "to_v", "to_out.0"]
    
    # Training
    batch_size = 1
    learning_rate = 1e-4
    max_train_steps = 3000             # 1000, 3000 or 5000
    gradient_accumulation_steps = 1
    mixed_precision = "fp16"
    seed = 42
    
    # Inference
    inference_prompt = "cinematic disney style character walking in forest"
    
    # Paths
    caption_file = os.path.join(local_dir, "captions.txt")
    lora_save_path = "./svd_video_lora"

config = Config()
torch.manual_seed(config.seed)

# %% [markdown]
# # 3. Download Dataset from Hugging Face

# %%
print("Downloading Disney Video Dataset...")
snapshot_download(
    repo_id=config.dataset_repo,
    repo_type="dataset",
    local_dir=config.local_dir
)
print("Download complete.")

# The dataset contains:
#   - video files in the root (e.g., 00000.mp4, 00001.mp4, ...)
#   - videos.txt  (list of video filenames)
#   - prompt.txt  (original captions, one per line)

# %%
# Read video paths and original captions
with open(os.path.join(config.local_dir, "videos.txt"), "r") as f:
    video_filenames = [line.strip() for line in f]
video_paths = [os.path.join(config.local_dir, v) for v in video_filenames]

with open(os.path.join(config.local_dir, "prompt.txt"), "r") as f:
    original_captions = [line.strip() for line in f]

print(f"Found {len(video_paths)} videos.")
print(f"Example video: {video_paths[0]}")
print(f"Example original caption: {original_captions[0]}")

# %% [markdown]
# # 4. (Optional) Improve Captions with BLIP
# 
# The dataset already provides captions, but we can re‑generate them with BLIP for better quality.
# Uncomment the following cell to run BLIP on all videos. This may take a while.

# %%
def generate_caption_with_blip(video_path, processor, model):
    """Extract middle frame from video and generate caption using BLIP."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return ""
    mid_frame = total_frames // 2
    cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        return ""
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(frame_rgb)
    inputs = processor(pil_image, return_tensors="pt").to(model.device)
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

# Uncomment to generate new captions (will overwrite existing captions.txt)
# processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
# blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("mps")
# new_captions = []
# for path in video_paths:
#     cap = generate_caption_with_blip(path, processor, blip_model)
#     new_captions.append(cap)
# with open(config.caption_file, "w") as f:
#     for c in new_captions:
#         f.write(c + "\n")
# print("BLIP captions saved.")

# For now, we use the original captions. You can later replace them with the improved ones.
captions = original_captions  # or load from config.caption_file if you generated new ones

# %% [markdown]
# # 5. Video Dataset Class (OpenCV)

# %%
class VideoDataset(Dataset):
    """Load videos, sample `num_frames` uniformly, resize to `size`."""
    def __init__(self, video_paths, captions, size=256, frames=16):
        self.video_paths = video_paths
        self.captions = captions
        self.size = size
        self.frames = frames

    def __len__(self):
        return len(self.video_paths)

    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total < self.frames:
            # If video is too short, duplicate frames
            indices = np.linspace(0, total-1, self.frames).astype(int)
        else:
            indices = np.linspace(0, total-1, self.frames).astype(int)
        
        frames = []
        current = 0
        target = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if target < len(indices) and current == indices[target]:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
                target += 1
            current += 1
        cap.release()
        
        if len(frames) == 0:
            raise ValueError(f"No frames loaded from {path}")
        
        # Pad if we got fewer frames (should not happen with linspace)
        while len(frames) < self.frames:
            frames.append(frames[-1])
        
        frames = np.array(frames).astype(np.float32) / 255.0
        frames = torch.from_numpy(frames).permute(0, 3, 1, 2)  # [F, C, H, W]
        frames = F.interpolate(frames, size=(self.size, self.size), mode="bilinear", align_corners=False)
        return frames

    def __getitem__(self, idx):
        frames = self.load_video(self.video_paths[idx])
        return {"pixel_values": frames, "caption": self.captions[idx]}

# %%
dataset = VideoDataset(video_paths, captions, size=config.size, frames=config.num_frames)
dataloader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)

# %% [markdown]
# # 6. Load Base Model and Inject LoRA

# %%
# Load SVD pipeline
pipe = StableVideoDiffusionPipeline.from_pretrained(
    config.base_model,
    torch_dtype=torch.float16,
    variant="fp16"
)
# Replace scheduler with training-friendly DDPM scheduler
pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

# Freeze VAE and image encoder
pipe.vae.requires_grad_(False)
pipe.image_encoder.requires_grad_(False)

# Add LoRA to UNet
lora_config = LoraConfig(
    r=config.lora_rank,
    lora_alpha=config.lora_alpha,
    target_modules=config.target_modules,
    init_lora_weights="gaussian"
)
pipe.unet = get_peft_model(pipe.unet, lora_config)
pipe.unet.enable_gradient_checkpointing()   # memory saving

print("LoRA injected. Trainable parameters:", sum(p.numel() for p in pipe.unet.parameters() if p.requires_grad))

# %% [markdown]
# # 7. Prepare Optimizer and Accelerator

# %%
optimizer = torch.optim.AdamW(pipe.unet.parameters(), lr=config.learning_rate)

accelerator = Accelerator(
    mixed_precision=config.mixed_precision,
    gradient_accumulation_steps=config.gradient_accumulation_steps
)
pipe.unet, optimizer, dataloader = accelerator.prepare(pipe.unet, optimizer, dataloader)

# Move VAE and image encoder to the accelerator device (they are frozen)
pipe.vae.to(accelerator.device)
pipe.image_encoder.to(accelerator.device)

# %% [markdown]
# # 8. Training Loop


# %%
global_step = 0
print("Starting training...")
for epoch in range(2):  # will break after max steps
    for batch in dataloader:
        with accelerator.accumulate(pipe.unet):
            # Prepare pixel values: [bs, frames, C, H, W] in [0,1] -> [-1,1]
            pixel_vals = batch["pixel_values"].to(dtype=torch.float16)
            pixel_vals = pixel_vals * 2.0 - 1.0
            bs, num_f, c, h, w = pixel_vals.shape

            # First frame for conditioning
            first_frame = pixel_vals[:, 0, :, :, :]  # [bs, C, H, W]

            # Image embeddings from CLIP
            img_for_clip = F.interpolate(first_frame, size=(224,224), mode='bilinear')
            with torch.no_grad():
                image_embeds = pipe.image_encoder(img_for_clip).image_embeds.unsqueeze(1)  # [bs,1,1024]

            # Encode video latents
            flat_pixels = pixel_vals.view(bs * num_f, c, h, w)
            with torch.no_grad():
                latents = pipe.vae.encode(flat_pixels).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
            latents = latents.view(bs, num_f, 4, h//8, w//8)

            # Encode first frame latents (condition)
            with torch.no_grad():
                cond_latents = pipe.vae.encode(first_frame).latent_dist.sample()
            cond_latents = cond_latents * pipe.vae.config.scaling_factor
            cond_latents = cond_latents.unsqueeze(1).repeat(1, num_f, 1, 1, 1)

            # Noise and timesteps
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bs,), device=latents.device).long()
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)

            # Concatenate noisy latents and condition latents (channel dim)
            model_input = torch.cat([noisy_latents, cond_latents], dim=2)  # [bs, num_f, 8, h//8, w//8]

            # Time IDs (fps, motion_bucket, noise_aug) – fixed values for training
            added_time_ids = torch.tensor([[7.0, 127.0, 0.02]], device=latents.device).repeat(bs, 1)

            # Forward
            model_pred = pipe.unet(
                model_input,
                timesteps,
                encoder_hidden_states=image_embeds,
                added_time_ids=added_time_ids
            ).sample

            loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

            global_step += 1
            if global_step % 50 == 0:
                print(f"Step {global_step} | Loss: {loss.item():.4f}")

            if global_step >= config.max_train_steps:
                break
    if global_step >= config.max_train_steps:
        break

print("Training finished.")

# %%
# Save the LoRA weights
accelerator.unwrap_model(pipe.unet).save_pretrained(config.lora_save_path)
print(f"LoRA saved to {config.lora_save_path}")

# %% [markdown]
# # 9. Evaluation Metrics (CLIPScore, LPIPS)

# %%
# Initialize metrics
clip_score_fn = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16")
lpips_fn = LearnedPerceptualImagePatchSimilarity(net_type='alex')

def evaluate_video(video_frames, prompt):
    """
    video_frames: list of PIL Images or numpy arrays (0-255)
    prompt: text prompt
    Returns: (clip_score, lpips_score)
    """
    # Convert frames to PIL if necessary
    pil_frames = []
    for frame in video_frames:
        if isinstance(frame, np.ndarray):
            pil_frames.append(Image.fromarray(frame))
        else:
            pil_frames.append(frame)
    
    # CLIPScore: average over frames
    clip_scores = []
    for frame in pil_frames:
        clip_scores.append(clip_score_fn(frame, [prompt]).item())
    avg_clip = np.mean(clip_scores)
    
    # LPIPS: average pairwise distance between consecutive frames
    lpips_vals = []
    for i in range(len(pil_frames)-1):
        img1 = transforms.ToTensor()(pil_frames[i]).unsqueeze(0) * 2 - 1   # [-1,1]
        img2 = transforms.ToTensor()(pil_frames[i+1]).unsqueeze(0) * 2 - 1
        lpips_vals.append(lpips_fn(img1, img2).item())
    avg_lpips = np.mean(lpips_vals) if lpips_vals else 0.0
    
    return avg_clip, avg_lpips

# %% [markdown]
# # 10. Inference Pipeline (Text → Video)
# 
# We use Stable Diffusion 1.5 to generate the first image from a prompt, then feed it to the fine‑tuned SVD to produce the video.

# %%
# Load SD 1.5 for text-to-image (offloaded to CPU when not used)
sd_pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
sd_pipe.to("cpu")  # start on CPU

# Load the fine-tuned LoRA into the SVD pipeline
# We need to re-load the base model and then attach LoRA
pipe = StableVideoDiffusionPipeline.from_pretrained(
    config.base_model,
    torch_dtype=torch.float16,
    variant="fp16"
)
pipe.unet = PeftModel.from_pretrained(pipe.unet, config.lora_save_path)
pipe.scheduler = DDPMScheduler.from_config(pipe.scheduler.config)
pipe.vae.requires_grad_(False)
pipe.image_encoder.requires_grad_(False)

def text_to_video(prompt, num_frames=16, fps=7):
    # Step 1: Generate initial image with SD
    sd_pipe.to("mps")
    init_image = sd_pipe(prompt).images[0].resize((config.size, config.size))
    sd_pipe.to("cpu")
    gc.collect()
    torch.mps.empty_cache()
    
    # Step 2: Generate video with SVD
    pipe.to("mps")
    video_frames = pipe(
        image=init_image,
        num_frames=num_frames,
        height=config.size,
        width=config.size,
        decode_chunk_size=4
    ).frames[0]
    pipe.to("cpu")
    gc.collect()
    torch.mps.empty_cache()
    
    return video_frames, init_image

# %%
# Test inference
frames, first_img = text_to_video(config.inference_prompt, num_frames=config.num_frames, fps=config.fps)
export_to_video(frames, "output.mp4", fps=config.fps)
print("Video saved as output.mp4")
display(Video("output.mp4", embed=True, width=512))

# Evaluate
clip, lpips = evaluate_video(frames, config.inference_prompt)
print(f"CLIPScore: {clip:.3f} (target ≥0.25), LPIPS: {lpips:.3f} (target ≤0.30)")

# %% [markdown]
# # 11. Gradio Demo: Upload, Edit Caption, Log Preferences
# 
# This interface allows you to upload a custom video, see its auto‑generated caption (using BLIP), edit it, and save the pair to a preference log (`preferences.csv`).

# %%
def process_upload(video_path):
    """Generate caption for uploaded video using BLIP."""
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("mps")
    
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return video_path, "Could not read video."
    cap.set(cv2.CAP_PROP_POS_FRAMES, total_frames//2)
    ret, frame = cap.read()
    cap.release()
    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(frame_rgb)
        inputs = processor(pil_image, return_tensors="pt").to("mps")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
    else:
        caption = "Failed to extract frame."
    return video_path, caption

def save_preference(video_path, edited_caption):
    """Append the (video path, caption) to a CSV file."""
    with open("preferences.csv", "a") as f:
        f.write(f"{video_path},{edited_caption}\n")
    return "Preference saved!"

# Build Gradio interface
with gr.Blocks(title="Disney Video Caption Editor") as demo:
    gr.Markdown("## Upload a video, edit its caption, and log as preferred")
    with gr.Row():
        video_input = gr.Video(label="Upload your video")
        caption_output = gr.Textbox(label="Auto-generated caption", lines=3)
    with gr.Row():
        edit_box = gr.Textbox(label="Edit caption if needed", lines=3)
        log_btn = gr.Button("Log as preferred")
    status = gr.Textbox(label="Status")
    
    video_input.change(fn=process_upload, inputs=video_input, outputs=[video_input, caption_output])
    caption_output.change(lambda x: x, inputs=caption_output, outputs=edit_box)  # copy to edit box
    log_btn.click(fn=save_preference, inputs=[video_input, edit_box], outputs=status)

# To launch the demo, uncomment the next line:
# demo.launch()

# %% [markdown]
# # 12. Experimentation Notes
# 
# You can easily change the following hyperparameters in the `Config` class:
# - `lora_rank`: 4, 8, 16
# - `max_train_steps`: 1000, 3000, 5000
# - `size`: 256 or 512 (but 512 requires more memory)
# - `target_modules`: For AnimateDiff, you would change to motion module targets.
# 
# To try **AnimateDiff** instead of SVD, replace the pipeline and adjust the training loop accordingly (different conditioning). The LoRA injection points would then be the motion modules.
# 
# The preference log (`preferences.csv`) can be used to curate a higher‑quality dataset for future fine‑tuning.

# %% [markdown]
# # 13. Cleanup
# 
# Free memory after all cells.

# %%
del pipe, sd_pipe
gc.collect()
torch.mps.empty_cache()
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchmetrics] [torchmetrics]


Fetching 75 files: 100%|██████████| 75/75 [00:00<00:00, 2964.90it/s]


Download complete.
Found 69 videos.
Example video: ./video_dataset/videos/a3c275fc2eb0a67168a7c58a6a9adb14.mp4
Example original caption: A black and white animated scene unfolds with an anthropomorphic goat surrounded by musical notes and symbols, suggesting a playful environment. Mickey Mouse appears, leaning forward in curiosity as the goat remains still. The goat then engages with Mickey, who bends down to converse or react. The dynamics shift as Mickey grabs the goat, potentially in surprise or playfulness, amidst a minimalistic background. The scene captures the evolving relationship between the two characters in a whimsical, animated setting, emphasizing their interactions and emotions.


Loading pipeline components...: 100%|██████████| 5/5 [00:01<00:00,  4.03it/s]


LoRA injected. Trainable parameters: 3319808
Starting training...
Step 50 | Loss: 0.9588
